# Sentiment Prediction using Simple RNN

This notebook loads a trained model and performs sentiment prediction on custom input text.

Pipeline:

$$
\text{User Text} \rightarrow \text{Tokenization} \rightarrow \text{Padding} \rightarrow \text{Model} \rightarrow \text{Prediction}
$$

In [21]:
# Step 1: Import Libraries and Load the Model
import numpy as np
import tensorflow as tf
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing import sequence
from tensorflow.keras.models import load_model

## Step 2: Load Word Index

We use the IMDB word index to map words to integers:

$$
\text{word} \rightarrow \text{index}
$$

In [22]:
# Load the IMDB dataset word index
word_index = imdb.get_word_index()

reverse_word_index = {value: key for key, value in word_index.items()}

## Step 3: Load Trained Model

We load the trained RNN model:

$$
\text{Model}: f(x) \rightarrow y
$$

Where:
- $x$ = input sequence  
- $y$ = sentiment probability  

In [23]:
# Load the pre-trained model

model = load_model("model/simple_rnn_imdb.keras")
model.summary()

Model: "sequential_7"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_7 (Embedding)         │ (None, 500, 128)       │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_7 (SimpleRNN)        │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,939,077 (15.03 MB)

 Trainable params: 1,313,025 (5.01 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 2,626,052 (10.02 MB)

## Step 4: Model Weights

Inspect learned weights:

$$
W = \text{Embedding weights + RNN weights + Dense weights}
$$

In [24]:
model.get_weights()

[array([[-0.01838003,  0.0241112 , -0.0220237 , ..., -0.03616286,
         -0.0334604 , -0.02911438],
        [ 0.01716308, -0.00219988, -0.04814647, ..., -0.03608379,
         -0.01880225, -0.04693761],
        [ 0.04008193,  0.02477326, -0.01162159, ...,  0.00645952,
          0.01356837, -0.01306073],
        ...,
        [ 0.05346512, -0.01809264,  0.0196225 , ...,  0.01418609,
         -0.03920563, -0.02307346],
        [-0.03489943,  0.00182826,  0.00445052, ..., -0.01778721,
          0.05001245,  0.00552908],
        [ 0.01145996,  0.02771742,  0.00021508, ...,  0.01280314,
          0.0308776 , -0.00550841]], shape=(10000, 128), dtype=float32),
 array([[-0.11211186,  0.07259472, -0.11194988, ...,  0.04204831,
         -0.00628635,  0.03599252],
        [-0.12772767, -0.05713225, -0.02888233, ..., -0.07187737,
         -0.08500296,  0.11630046],
        [-0.03713711,  0.07771657,  0.01681512, ...,  0.05743559,
         -0.10046948,  0.03076621],
        ...,
        [-0.1667091

## Step 5: Helper Functions

We define:

- Decode function → converts integers to words  
- Preprocessing function → converts text into model input  

$$
\text{Text} \rightarrow \text{Indices} \rightarrow \text{Padded Sequence}
$$

In [25]:
# Step 2: Helper Functions
# Function to decode reviews
def decode_review(encoded_review):
    return ' '.join([reverse_word_index.get(i - 3, '?') for i in encoded_review])

# Function to preprocess user input
def preprocess_text(text):
    words = text.lower().split()
    encoded_review = [word_index.get(word, 2) + 3 for word in words]
    padded_review = sequence.pad_sequences([encoded_review], maxlen=500)
    return padded_review

## Step 6: Prediction Function

Model outputs probability:

$$
p = P(\text{positive})
$$

Decision rule:

$$
\text{Sentiment} =
\begin{cases}
\text{Positive} & p > 0.5 \\
\text{Negative} & p \leq 0.5
\end{cases}
$$

In [26]:
### Prediction  function

def predict_sentiment(review):
    preprocessed_input=preprocess_text(review)

    prediction=model.predict(preprocessed_input)

    sentiment = 'Positive' if prediction[0][0] > 0.5 else 'Negative'
    
    return sentiment, prediction[0][0]

## Step 7: Test Prediction

We pass a custom input:

$$
\text{Input Review} \rightarrow \text{Model} \rightarrow \text{Prediction}
$$

In [27]:
# Example review for prediction
example_review = "This movie was fantastic! The acting was great and the plot was thrilling."

sentiment, score = predict_sentiment(example_review)

print(f'Review: {example_review}')
print(f'Sentiment: {sentiment}')
print(f'Prediction Score: {score}')

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 86ms/step
Review: This movie was fantastic! The acting was great and the plot was thrilling.
Sentiment: Positive
Prediction Score: 0.5039289593696594
